In [2]:
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv

load_dotenv()

#uso las credenciales donde esta mi gold
DB_USER = os.getenv('POSTGRES_USER')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD')
DB_HOST = 'localhost'
DB_PORT = 5432
DB_NAME = os.getenv('POSTGRES_DB')

#creo conexion
DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

#el engine
engine = create_engine(DATABASE_URL)

#config extra
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"Engine creado correctamente para base de datos: {DB_NAME}")

Engine creado correctamente para base de datos: ny_taxi


In [37]:
# 1. Viajes por mes (2024): count(*) por month.
# Tablas: fct_trips, dim_date

query = """
SELECT 
    d.month,
    d.month_name,
    COUNT(*) AS total_trips
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.month, d.month_name
ORDER BY d.month;
"""

df = pd.read_sql(query, engine)
print(df)

    month month_name  total_trips
0       1  January        2809917
1       2  February       2801682
2       3  March          3126318
3       4  April          3076185
4       5  May            3288778
5       6  June           3096092
6       7  July           2768093
7       8  August         2690940
8       9  September      3115128
9      10  October        3396651
10     11  November       3232903
11     12  December       3293408


El total de viajes mensual es similar en todos los meses del año pero el valor mas alto es en octubre y los niveles más bajos en agosto. En general el segundo semestre nos muestra algunos meses con mayor número de viajes que en los meses al inicio del año.

In [38]:
# 2. Viajes por service_type y mes.
# Tablas: fct_trips, dim_date, dim_service_type

query = """
SELECT 
    d.month,
    d.month_name,
    s.service_type_name,
    COUNT(*) AS total_trips
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_service_type s
    ON f.service_type_key = s.service_type_key
WHERE d.year = 2024
GROUP BY d.month, d.month_name, s.service_type_name
ORDER BY d.month, s.service_type_name;
"""

df = pd.read_sql(query, engine)
print(df)

    month month_name service_type_name  total_trips
0       1  January          Green Taxi        52446
1       1  January         Yellow Taxi      2757471
2       2  February         Green Taxi        49943
3       2  February        Yellow Taxi      2751739
4       3  March            Green Taxi        54601
5       3  March           Yellow Taxi      3071717
6       4  April            Green Taxi        53834
7       4  April           Yellow Taxi      3022351
8       5  May              Green Taxi        58347
9       5  May             Yellow Taxi      3230431
10      6  June             Green Taxi        52125
11      6  June            Yellow Taxi      3043967
12      7  July             Green Taxi        49523
13      7  July            Yellow Taxi      2718570
14      8  August           Green Taxi        49494
15      8  August          Yellow Taxi      2641446
16      9  September        Green Taxi        51974
17      9  September       Yellow Taxi      3063154
18     10  O

Los viajes en taxis amarillos son mucho mayores en todos los meses y green solo representa una pequeña parte de los viajes del mes. Esto nos indica que el servicio dominante es el amarillo en todos los meses del 2024. Yellow y green por separado mantienen un porcentaje promedio similar a los otros meses del mismo servicio indicando que es un patrón común. 

In [39]:
# 3. Top 10 zonas de pickup (total 2024).
# Tablas: fct_trips, dim_zone, dim_date

query = """
SELECT 
    z.zone AS pickup_zone,
    z.borough AS pickup_borough,
    COUNT(*) AS total_pickups
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z
    ON f.pu_zone_key = z.zone_key
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY z.zone, z.borough
ORDER BY total_pickups DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)
print(df)

                    pickup_zone pickup_borough  total_pickups
0                   JFK Airport         Queens        1894757
1         Upper East Side South      Manhattan        1777442
2                Midtown Center      Manhattan        1750072
3         Upper East Side North      Manhattan        1570913
4                  Midtown East      Manhattan        1320633
5  Penn Station/Madison Sq West      Manhattan        1280841
6     Times Sq/Theatre District      Manhattan        1265788
7             LaGuardia Airport         Queens        1258558
8           Lincoln Square East      Manhattan        1191174
9                 Midtown North      Manhattan        1072093


Los pickups que más viajes muestran son en zonas que normalmente tienen bastante demanda y necesidad de transporte fijándonos en la cultura o normalidad de Nueva York. Se destacan zonas como el aeropuerto que tiene sentido dado que la gente necesita un transporte de ahí a sus hoteles o casas también en varias zonas turisticas y centrales. 

In [40]:
# 4. Top 10 zonas de dropoff (total 2024).
# Tablas: fct_trips, dim_zone, dim_date

query = """
SELECT 
    z.zone AS dropoff_zone,
    z.borough AS dropoff_borough,
    COUNT(*) AS total_dropoffs
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z
    ON f.do_zone_key = z.zone_key
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY z.zone, z.borough
ORDER BY total_dropoffs DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)
print(df)

                dropoff_zone dropoff_borough  total_dropoffs
0      Upper East Side North       Manhattan         1680938
1      Upper East Side South       Manhattan         1595661
2             Midtown Center       Manhattan         1386143
3  Times Sq/Theatre District       Manhattan         1162369
4                Murray Hill       Manhattan         1081651
5               Midtown East       Manhattan         1064569
6        Lincoln Square East       Manhattan         1038248
7      Upper West Side South       Manhattan         1036720
8            Lenox Hill West       Manhattan          969021
9               East Chelsea       Manhattan          936287


Los dropoffs se concentran principalmente en Manhattan o en sus alrededores lo que nos indica que la gente se queda en zonas turisticas y residenciales que tienen bastante actividad en Nueva York. Igual tomando el top 10 Manhattan es el destino número uno de los viajes. 

In [42]:
# 5. Top 5 boroughs por mes (pickup).
# Tablas: fct_trips, dim_zone, dim_date

query = """
WITH borough_month AS (
    SELECT
        d.month,
        d.month_name,
        z.borough,
        COUNT(*) AS trips
    FROM analytics_gold.fct_trips f
    JOIN analytics_gold.dim_zone z
        ON f.pu_zone_key = z.zone_key
    JOIN analytics_gold.dim_date d
        ON f.pickup_date_key = d.date_key
    WHERE d.year = 2024
    GROUP BY d.month, d.month_name, z.borough
),
ranked AS (
    SELECT
        month,
        month_name,
        borough,
        trips,
        ROW_NUMBER() OVER (PARTITION BY month ORDER BY trips DESC) AS row_n
    FROM borough_month
)
SELECT
    month,
    month_name,
    borough,
    trips
FROM ranked
WHERE row_n <= 5
ORDER BY month, trips DESC;
"""

df = pd.read_sql(query, engine)
print(df.head(20))
print(df.shape)

    month month_name    borough    trips
0       1  January    Manhattan  2494528
1       1  January       Queens   273049
2       1  January     Brooklyn    24587
3       1  January      Unknown    11101
4       1  January        Bronx     6322
5       2  February   Manhattan  2508699
6       2  February      Queens   250116
7       2  February    Brooklyn    26099
8       2  February     Unknown    10341
9       2  February       Bronx     6108
10      3  March      Manhattan  2766353
11      3  March         Queens   307858
12      3  March       Brooklyn    32254
13      3  March        Unknown    12139
14      3  March          Bronx     7315
15      4  April      Manhattan  2714128
16      4  April         Queens   309864
17      4  April       Brooklyn    32752
18      4  April        Unknown    11514
19      4  April          Bronx     7373
(60, 4)


Manhattan es el primero en todos los meses en número de pickups con Queens como segundo lugar que nuevamente podemos ver que son zonas con alto flujo de gente a diario y también zonas bastante turisticas, los demás boroughs quedan de hecho bastante por debajo de estos dos.

In [44]:
# 6. Horas pico (top 5 horas) para cada día de semana.
# Tablas: fct_trips, dim_date

query = """
WITH base AS (
    SELECT
        d.day_of_week,
        d.day_name,
        EXTRACT(HOUR FROM f.pickup_datetime) AS pickup_hour,
        COUNT(*) AS trips
    FROM analytics_gold.fct_trips f
    JOIN analytics_gold.dim_date d
        ON f.pickup_date_key = d.date_key
    WHERE d.year = 2024
    GROUP BY d.day_of_week, d.day_name, EXTRACT(HOUR FROM f.pickup_datetime)
),
ranked AS (
    SELECT
        day_of_week,
        day_name,
        pickup_hour,
        trips,
        ROW_NUMBER() OVER (PARTITION BY day_of_week ORDER BY trips DESC) AS row_n
    FROM base
)
SELECT
    day_of_week,
    day_name,
    pickup_hour,
    trips
FROM ranked
WHERE row_n <= 5
ORDER BY day_of_week, trips DESC;
"""

df = pd.read_sql(query, engine)
print(df)


    day_of_week   day_name  pickup_hour   trips
0             0  Sunday            16.0  296642
1             0  Sunday            14.0  295160
2             0  Sunday            17.0  293610
3             0  Sunday            15.0  288260
4             0  Sunday            13.0  283545
5             1  Monday            18.0  345142
6             1  Monday            17.0  339575
7             1  Monday            15.0  320023
8             1  Monday            16.0  316198
9             1  Monday            14.0  310258
10            2  Tuesday           18.0  407550
11            2  Tuesday           17.0  385428
12            2  Tuesday           19.0  350088
13            2  Tuesday           16.0  345000
14            2  Tuesday           15.0  342215
15            3  Wednesday         18.0  419514
16            3  Wednesday         17.0  395392
17            3  Wednesday         19.0  364844
18            3  Wednesday         16.0  354164
19            3  Wednesday         21.0 

Las horas pico que podemos ver son principalmente en la tarde/noche de todos los días como desde las dos de la tarde hasta las nueve de la noche, tiene lógica porque en estas horas son las salidas de trabajos y las personas se encontrarían regresando a sus casas. 

In [45]:
# 7. Distribución de viajes por día de semana (ranking).
# Tablas: fct_trips, dim_date

query = """
SELECT 
    d.day_of_week,
    d.day_name,
    COUNT(*) AS total_trips,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.day_of_week, d.day_name
ORDER BY total_trips DESC;
"""

df = pd.read_sql(query, engine)
print(df)

   day_of_week   day_name  total_trips  percentage
0            4  Thursday       5750581       15.67
1            3  Wednesday      5528143       15.06
2            5  Friday         5444990       14.84
3            6  Saturday       5382211       14.67
4            2  Tuesday        5356203       14.60
5            1  Monday         4697634       12.80
6            0  Sunday         4536333       12.36


La mayor cantidad de viajes son jueves, viernes y miércoles que son días de más actividad por empiezo de fin de semana y días de mayor actividad en general. Domingo en cambio es el día de menos cantidad de viajes que no es un día laboral y normalmente de descanso. 

In [46]:
# 8. Ingreso total (total_amount) por mes.
# Tablas: fct_trips, dim_date

query = """
SELECT
    d.month,
    d.month_name,
    SUM(f.total_amount) AS total_amount
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.month, d.month_name
ORDER BY d.month;
"""

df = pd.read_sql(query, engine)
print(df)

    month month_name  total_amount
0       1  January    7.708523e+07
1       2  February   7.679230e+07
2       3  March      8.861406e+07
3       4  April      8.840243e+07
4       5  May        9.698528e+07
5       6  June       9.027528e+07
6       7  July       8.170355e+07
7       8  August     8.010742e+07
8       9  September  9.433275e+07
9      10  October    1.016998e+08
10     11  November   9.370740e+07
11     12  December   9.729939e+07


El ingreso total sigue una tendencia similar al volumen de viajes por mes que ya vimos teniendo el valor de ganancia más alto en el mes de octubre y niveles altos también en meses como diciembre, noviembre, mayo y junio. Los primeros meses del año tienen los menores ingresos siendo estos enero y febrero. 

In [48]:
# 9. Ingreso total por service_type y mes.
# Tablas: fct_trips, dim_date, dim_service_type

query = """
SELECT
    d.month,
    d.month_name,
    s.service_type_name,
    SUM(f.total_amount) AS total_revenue
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_service_type s
    ON f.service_type_key = s.service_type_key
WHERE d.year = 2024
GROUP BY d.month, d.month_name, s.service_type_name
ORDER BY d.month, total_revenue DESC;
"""

df = pd.read_sql(query, engine)
print(df)

    month month_name service_type_name  total_revenue
0       1  January         Yellow Taxi   7.593164e+07
1       1  January          Green Taxi   1.153595e+06
2       2  February        Yellow Taxi   7.567656e+07
3       2  February         Green Taxi   1.115746e+06
4       3  March           Yellow Taxi   8.737294e+07
5       3  March            Green Taxi   1.241115e+06
6       4  April           Yellow Taxi   8.715335e+07
7       4  April            Green Taxi   1.249073e+06
8       5  May             Yellow Taxi   9.555969e+07
9       5  May              Green Taxi   1.425597e+06
10      6  June            Yellow Taxi   8.898785e+07
11      6  June             Green Taxi   1.287429e+06
12      7  July            Yellow Taxi   8.048023e+07
13      7  July             Green Taxi   1.223327e+06
14      8  August          Yellow Taxi   7.882382e+07
15      8  August           Green Taxi   1.283601e+06
16      9  September       Yellow Taxi   9.294820e+07
17      9  September        

El ingreso mensual viene en su mayoria del servicio yellow y un poco de aporte por parte del servicio green, igual podemos ver que yellow domina en todos los meses y que los ingresos provienen casi en su totalidad por parte de green.  

In [49]:
# 10. tip % promedio por mes (avg(tip_amount / nullif(fare_amount,0))).
# Tablas: fct_trips, dim_date

query = """
SELECT
    d.month,
    d.month_name,
    AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.month, d.month_name
ORDER BY d.month;
"""

df = pd.read_sql(query, engine)
print(df)

    month month_name   avg_tip
0       1  January    0.232434
1       2  February   0.217524
2       3  March      0.216806
3       4  April      0.221161
4       5  May        0.219660
5       6  June       0.209523
6       7  July       0.209608
7       8  August     0.202547
8       9  September  0.213758
9      10  October    0.213893
10     11  November   0.228334
11     12  December   0.211249


El porcentaje promedio de propina se mantiene bastante estable en todos los meses calrededor del 20 porciento o hasta un poco más lo que cumple con la espectativa o costumbre que se tiene en Estados Unidos. Enero y noviembre tienen los valores más altos de las propinas y agosto cuenta con el menor de todos. 

In [51]:
# 11. tip % por borough y mes.
# Tablas: fct_trips, dim_date, dim_zone

query = """
SELECT
    d.month,
    d.month_name,
    z.borough,
    AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_zone z
    ON f.pu_zone_key = z.zone_key
WHERE d.year = 2024
GROUP BY d.month, d.month_name, z.borough
ORDER BY d.month, avg_tip DESC;
"""

df = pd.read_sql(query, engine)
print(df)

    month month_name    borough   avg_tip
0       1  January      Unknown  4.273114
1       1  January          EWR  0.488626
2       1  January    Manhattan  0.223994
3       1  January       Queens  0.162120
4       1  January     Brooklyn  0.105147
..    ...        ...        ...       ...
79     12  December     Unknown  0.211854
80     12  December         EWR  0.196757
81     12  December      Queens  0.166710
82     12  December    Brooklyn  0.092058
83     12  December       Bronx  0.036260

[84 rows x 4 columns]


En esta consulta nos encontramos con categorías como unknown con valores anómalos que son probablemente por zonas mal clasificadas. De ahí los principales parecen ser Manhattan y EWR que los otros lugares. 

In [52]:
# 12. Top 10 zonas por ingreso total (pickup).
# Tablas: fct_trips, dim_date, dim_zone

query = """
SELECT
    z.zone AS pickup_zone,
    z.borough AS pickup_borough,
    SUM(f.total_amount) AS total_amount
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_zone z
    ON f.pu_zone_key = z.zone_key
WHERE d.year = 2024
GROUP BY z.zone, z.borough
ORDER BY total_amount DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)
print(df)

                    pickup_zone pickup_borough  total_amount
0                   JFK Airport         Queens  1.557755e+08
1             LaGuardia Airport         Queens  8.611638e+07
2                Midtown Center      Manhattan  4.450841e+07
3     Times Sq/Theatre District      Manhattan  3.718472e+07
4         Upper East Side South      Manhattan  3.711052e+07
5         Upper East Side North      Manhattan  3.300421e+07
6  Penn Station/Madison Sq West      Manhattan  3.286899e+07
7                  Midtown East      Manhattan  3.261632e+07
8                 Midtown North      Manhattan  2.729234e+07
9           Lincoln Square East      Manhattan  2.658337e+07


Las zonas que tienen el mayor ingreso son ambos aeropuertos y areas igual como ya indicamos bastante visitadas por turistas como Times Sq. Esto nos indica que los aeropuertos y el centro turistico y más recurrido tienen los tickets más valiosos.

In [54]:
# 13. Top 10 zonas por tip % (pickup) con mínimo N viajes (define N y documenta)
# Tablas: fct_trips, dim_date, dim_zone

query = """
WITH zone_stats AS (
    SELECT
        z.zone AS pickup_zone,
        z.borough AS pickup_borough,
        COUNT(*) AS trips,
        AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip
    FROM analytics_gold.fct_trips f
    JOIN analytics_gold.dim_date d
        ON f.pickup_date_key = d.date_key
    JOIN analytics_gold.dim_zone z
        ON f.pu_zone_key = z.zone_key
    WHERE d.year = 2024
    GROUP BY z.zone, z.borough
)
SELECT
    pickup_zone,
    pickup_borough,
    trips,
    avg_tip
FROM zone_stats
WHERE trips >= 200
ORDER BY avg_tip DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)
print(df)

                pickup_zone pickup_borough  trips   avg_tip
0            Outside of NYC        Unknown  21279  5.998641
1  Williamsbridge/Olinville          Bronx   3036  2.092219
2              Baisley Park         Queens  13338  0.878628
3                   Bedford       Brooklyn   7048  0.761337
4                  Red Hook       Brooklyn   8090  0.683448
5           Carroll Gardens       Brooklyn   3838  0.436837
6          Hamilton Heights      Manhattan  25057  0.387572
7            Newark Airport            EWR   5709  0.323233
8               Boerum Hill       Brooklyn  15947  0.308889
9              Forest Hills         Queens  37763  0.300637


Primero utilizamos en valor N de 200 para no contar con zonas que no tienen muchos viajes pero razonable dentro de loa valores que ya conocíamos. Este ranking nos muestra que las zonas con mayor propina que tienen un volumen de viajes mayor a 200 y hay casos o categorías no estándar que tienen valores extremos y podría ser por casos atípicos. 

In [58]:
# 14. Comparación cash vs card: viajes, ingreso total, tip %.
# Tablas: fct_trips, dim_date, dim_payment_type

query = """
SELECT 
    p.payment_type_name,
    COUNT(*) AS total_trips,
    ROUND(SUM(f.total_amount), 2) AS total_amount,
    ROUND(AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) * 100, 2) AS avg_tip
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_payment_type p
    ON f.payment_type_key = p.payment_type_key
WHERE p.payment_type_name IN ('Credit Card', 'Cash')
GROUP BY p.payment_type_name
ORDER BY total_trips DESC;
"""

df = pd.read_sql(query, engine)
print(df)

  payment_type_name  total_trips  total_amount  avg_tip
0       Credit Card     30579619  9.137734e+08    25.97
1              Cash      5498069  1.364431e+08     0.00


Los pagos con Credit Card concentran la gran mayoría de los viajes en el 2024 y del ingreso total lo que es normal dado el año y que la gente ya no usa efectivo y pagan todo principalmente con tarjeta. La propina promedio es mucho mayor en tarjeta mientras que en Cash o efectivo es prácticamente nula se puede dar por fallas en el registro. 

In [59]:
# 15. Duración promedio (min) por mes.
# Tablas: fct_trips, dim_date

query = """
SELECT
    d.month,
    d.month_name,
    AVG(f.trip_duration_min) AS avg_duration_min
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
  AND f.trip_duration_min > 0
GROUP BY d.month, d.month_name
ORDER BY d.month;
"""

df = pd.read_sql(query, engine)
print(df)

    month month_name  avg_duration_min
0       1  January           15.703348
1       2  February          16.071551
2       3  March             16.811227
3       4  April             17.174916
4       5  May               18.171432
5       6  June              17.731409
6       7  July              17.395195
7       8  August            17.543060
8       9  September         18.770937
9      10  October           18.354244
10     11  November          17.882173
11     12  December          18.735076


La duración promedio de los viajes empieza más baja de lo que acaba y alcanza sus valores más altos al final del año y em mayo. Esto nos indica que los trayectos más largos o mayor tráfico se dio en la segunda mitad del año 2024.

In [60]:
# 16. Distancia promedio por mes.
# Tablas: fct_trips, dim_date

query = """
SELECT
    d.month,
    d.month_name,
    AVG(f.trip_distance) AS avg_distance
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
  AND f.trip_distance > 0
GROUP BY d.month, d.month_name
ORDER BY d.month;
"""

df = pd.read_sql(query, engine)
print(df)

    month month_name  avg_distance
0       1  January        3.302765
1       2  February       3.407958
2       3  March          3.570021
3       4  April          3.601540
4       5  May            3.462269
5       6  June           3.808741
6       7  July           3.729152
7       8  August         3.882275
8       9  September      3.580993
9      10  October        3.556734
10     11  November       3.337585
11     12  December       3.338130


La distancia promedio se mantiene bastante estable en todos los meses del 2024 en un rango de entre 3 y 3.8 millas. Los valores más altos aparecen en los meses de verano lo que nos indica que hubo una mayor proporción de trayectos más largos en distancia en ese periodo del año. 

In [61]:
# 17. Velocidad promedio (mph) por borough y franja horaria.
# Tablas: fct_trips, dim_date, dim_zone

query = """
WITH base AS (
    SELECT
        z.borough,
        CASE
            WHEN EXTRACT(HOUR FROM f.pickup_datetime) BETWEEN 6 AND 9 THEN 'AM_peak'
            WHEN EXTRACT(HOUR FROM f.pickup_datetime) BETWEEN 10 AND 15 THEN 'Midday'
            WHEN EXTRACT(HOUR FROM f.pickup_datetime) BETWEEN 16 AND 19 THEN 'PM_peak'
            ELSE 'Off_peak'
        END AS time_band,
        (f.trip_distance / NULLIF(f.trip_duration_min / 60.0, 0)) AS mph
    FROM analytics_gold.fct_trips f
    JOIN analytics_gold.dim_date d
        ON f.pickup_date_key = d.date_key
    JOIN analytics_gold.dim_zone z
        ON f.pu_zone_key = z.zone_key
    WHERE d.year = 2024
      AND f.trip_distance > 0
      AND f.trip_duration_min > 0
)
SELECT
    borough,
    time_band,
    AVG(mph) AS avg_mph
FROM base
GROUP BY borough, time_band
ORDER BY borough, time_band;
"""

df = pd.read_sql(query, engine)
print(df)

          borough time_band     avg_mph
0           Bronx   AM_peak   17.073511
1           Bronx    Midday   14.299289
2           Bronx  Off_peak   47.003358
3           Bronx   PM_peak   18.642327
4        Brooklyn   AM_peak   15.752191
5        Brooklyn    Midday   16.820400
6        Brooklyn  Off_peak   25.747111
7        Brooklyn   PM_peak   23.073559
8             EWR   AM_peak  414.133939
9             EWR    Midday  122.383824
10            EWR  Off_peak   69.515443
11            EWR   PM_peak  140.382445
12      Manhattan   AM_peak   11.264954
13      Manhattan    Midday    8.876772
14      Manhattan  Off_peak   11.646109
15      Manhattan   PM_peak    8.862349
16         Queens   AM_peak   25.799061
17         Queens    Midday   21.452602
18         Queens  Off_peak   29.341955
19         Queens   PM_peak   21.537756
20  Staten Island   AM_peak   26.145380
21  Staten Island    Midday   74.529714
22  Staten Island  Off_peak  109.445700
23  Staten Island   PM_peak  109.192274


Esta consulta nos indica que Manhattan tiene las velocidades promedio más bajas lo que es consistente con un mayor tráfico en la zona en la que es conocido que exista bastante movimiento de personas. En cambio Queens, Bronx o Staten Island presentan velocidades mayores o hasta categorías como EWR que nos muestra valores extremos que deberían tratarse como posibles outliers.

In [62]:
# 18. Percentiles p50 y p90 de duración por borough.
# Tablas: fct_trips, dim_date, dim_zone

query = """
SELECT
    z.borough,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY f.trip_duration_min) AS p50_duration_min,
    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY f.trip_duration_min) AS p90_duration_min
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_zone z
    ON f.pu_zone_key = z.zone_key
WHERE d.year = 2024
  AND f.trip_duration_min > 0
GROUP BY z.borough
ORDER BY p90_duration_min DESC;
"""

df = pd.read_sql(query, engine)
print(df)

         borough  p50_duration_min  p90_duration_min
0          Bronx         37.366667         81.683333
1       Brooklyn         23.283333         67.116667
2         Queens         33.883333         63.266667
3  Staten Island         12.566667         43.116667
4        Unknown         11.000000         33.533333
5      Manhattan         11.700000         26.616667
6            EWR          0.133333          1.333333


Bronx y Brooklyn presentan duraciones mayores que Manhattan lo que nos puede estar sugiriendo que hubo viajes más largos dentro de estas zonas. Manhattan tiene trayectos más cortos en ambos casos entre los boroughs principales que hemos mencionado bastante.

In [63]:
# 19. Top 10 zonas (pickup) por p90 de duración.
# Tablas: fct_trips, dim_date, dim_zone

query = """
SELECT
    z.zone AS pickup_zone,
    z.borough AS pickup_borough,
    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY f.trip_duration_min) AS p90_duration_min
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_zone z
    ON f.pu_zone_key = z.zone_key
WHERE d.year = 2024
  AND f.trip_duration_min > 0
GROUP BY z.zone, z.borough
ORDER BY p90_duration_min DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)
print(df)

       pickup_zone pickup_borough  p90_duration_min
0     Far Rockaway         Queens        109.558333
1  Hammels/Arverne         Queens        109.113333
2    Rockaway Park         Queens        104.956667
3     Coney Island       Brooklyn        104.360000
4     Crotona Park          Bronx         98.111667
5   Brighton Beach       Brooklyn         96.700000
6         Rosedale         Queens         95.135000
7       Co-Op City          Bronx         95.071667
8        Bronxdale          Bronx         94.743333
9        Gravesend       Brooklyn         93.566667


Las zonas que tienen el p90 de duración más alto estan principalmente en zonas alejadas como Hammels o Far Rockaway. Esto nos indica que aunque no son los viajes más largos en algunos casos si se pueden tardar, puede ser por distancia, tráficos u otros factores.  

In [65]:
# 20. Top 10 rutas borough→borough (pickup borough to dropoff borough) por número de viajes.
# Tablas: fct_trips, dim_date, dim_zone (pickup), dim_zone (dropoff)

query = """
SELECT
    zpu.borough AS pickup_borough,
    zdo.borough AS dropoff_borough,
    COUNT(*) AS trips
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_zone zpu
    ON f.pu_zone_key = zpu.zone_key
JOIN analytics_gold.dim_zone zdo
    ON f.do_zone_key = zdo.zone_key
WHERE d.year = 2024
GROUP BY zpu.borough, zdo.borough
ORDER BY trips DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)
print(df)

  pickup_borough dropoff_borough     trips
0      Manhattan       Manhattan  30240916
1         Queens       Manhattan   2084884
2      Manhattan          Queens   1006401
3         Queens          Queens    925611
4      Manhattan        Brooklyn    718003
5         Queens        Brooklyn    537878
6       Brooklyn        Brooklyn    247512
7      Manhattan         Unknown    134750
8      Manhattan           Bronx    120504
9         Queens         Unknown    119172


La ruta que va desde Manhattan y se quedan en Manhattan son las que más viajes tienen con bastante ventaja, lo que nos confirma que la demanda más alta es dentro del centro más turístico y poblado de Nueva York. También siguen viajes desde Queens o Bronx que confirman lo que pensamos hasta ahora y que se enfocan en el núcleo de la ciudad.  